## Versión final organizada

Este notebook calcula el CLE del TARGET xyz-control y escribe cachés, resultados y figuras dentro de `outputs/06_cle/target/xyz_control_nature_qiwenevents/`. La inspección detallada de las entradas está en `8_cse_qiwen_xyz_target.ipynb`; este notebook vuelve a cargarlas y validarlas para poder ejecutarse de forma independiente.


In [ ]:
import importlib
import project_paths as project_paths
project_paths = importlib.reload(project_paths)

EXPERIMENT_NAME = 'xyz_control_nature_qiwenevents'
CLE_TARGET_EXPERIMENT_ROOT, INPUT_DATA_DIR, FIGURES_DIR = (
    project_paths.cle_target_experiment_paths(EXPERIMENT_NAME)
)
RESULTS_DIR = CLE_TARGET_EXPERIMENT_ROOT / 'results'
INPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('TARGET inputs:', INPUT_DATA_DIR)
print('CLE results:', RESULTS_DIR)
print('CLE figures:', FIGURES_DIR)


# Controlled leading Lyapunov exponent for the Qiwen xyz-control TARGET

Este cálculo reproduce las intervenciones como **incrementos discretos del estado**. Para cada paso:

1. se aplica un paso libre RK4 de Lorenz–63;
2. se suma el impulso CSE almacenado;
3. el mismo impulso se aplica a la referencia y a todas las trayectorias perturbadas.

No se añade la intervención como fuerza continua dentro del RHS. El resultado es un LLE efectivo bajo una secuencia de intervenciones prescrita; no es la linealización completa del algoritmo realimentado de selección CSE y asimilación de datos.


## Imports


In [ ]:
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


## Load Validated CLE Inputs

Carga y valida de nuevo los tres archivos necesarios. No hace falta ejecutar previamente el notebook 8; ambos notebooks son independientes.


In [ ]:
from functools import partial

from class_cle_target import (
    load_and_validate_cse_target_inputs,
    rk4_free_step as lorenz63_rk4_free_step,
)

target_data = load_and_validate_cse_target_inputs(INPUT_DATA_DIR)
target_array = target_data['target_array']
intervention_array = target_data['intervention_array']
cse_summary = target_data['cse_summary']
Ta = target_data['Ta']
xdim = target_data['xdim']
dt = target_data['dt']
target_flat = target_data['target_flat']
intervention_flat = target_data['intervention_flat']
max_reconstruction_error = target_data['max_reconstruction_error']
rk4_free_step = partial(lorenz63_rk4_free_step, dt=dt)

print(
    'Validated CLE inputs:', target_array.shape,
    '| max reconstruction error:', max_reconstruction_error,
)


## CLE Configuration

`stored_discrete` replays the exact time-varying impulses. `fixed_discrete` is an optional comparison that repeats the first active impulse of the starting cycle after every free RK4 step. Both modes use discrete increments.

The calculations are checkpointed every 20 starting cycles. The no-renormalization cache stores only the ensemble-mean distance for each start, avoiding the approximately 1.1 GB per mode used by the historical notebook.


In [ ]:
requested_n_attractor_points_LE = 400
n_perturbations_LE = 100
n_steps_LE = 3500
perturbation_std_LE = 1e-8
FIT_START_LE = 0
FIT_END_LE = 1500
sampling_seed_LE = 42
perturbation_seed_LE = 2026
checkpoint_every_LE = 20
fixed_mode = 'first_active'
CACHE_VERSION_LE = 'qiwen_discrete_state_increment_v1'
CLE_MODES = ('stored_discrete', 'fixed_discrete')
RUN_NO_RENORM_DIAGNOSTIC = True
RUN_STEPWISE_RENORM = True

max_start_cycle = (target_flat.shape[0] - (n_steps_LE + 1)) // Ta
if max_start_cycle < 0:
    raise ValueError('The stored TARGET is shorter than the requested CLE horizon.')
candidate_cycle_ids = np.arange(max_start_cycle + 1, dtype=int)
n_attractor_points_LE = min(requested_n_attractor_points_LE, len(candidate_cycle_ids))

rng_sampling = np.random.default_rng(sampling_seed_LE)
sampled_cycle_ids = np.sort(
    rng_sampling.choice(candidate_cycle_ids, size=n_attractor_points_LE, replace=False)
)
rng_perturbation = np.random.default_rng(perturbation_seed_LE)
random_directions = rng_perturbation.uniform(
    -1.0,
    1.0,
    size=(n_attractor_points_LE, n_perturbations_LE, xdim),
)
random_directions /= np.clip(
    np.linalg.norm(random_directions, axis=2, keepdims=True),
    1e-300,
    None,
)
initial_perturbations = perturbation_std_LE * random_directions

print('Candidate cycle starts:', len(candidate_cycle_ids))
print('Sampled cycle starts:', n_attractor_points_LE)
print('Perturbations per start:', n_perturbations_LE)
print('Integration steps:', n_steps_LE)
print('Fit window:', FIT_START_LE, FIT_END_LE)
print('Modes:', CLE_MODES)


## Discrete Integration and Resumable Cache Helpers


In [ ]:
def intervention_sequence(mode, start_cycle, n_steps):
    start_index = int(start_cycle) * Ta
    if mode == 'stored_discrete':
        # u[n+1] advances TARGET[n] to TARGET[n+1].
        sequence = intervention_flat[start_index + 1:start_index + n_steps + 1]
        if len(sequence) != n_steps:
            raise ValueError('Not enough stored interventions for the requested horizon.')
        return sequence.copy()
    if mode == 'fixed_discrete':
        cycle_impulses = intervention_array[int(start_cycle)]
        active = np.where(np.linalg.norm(cycle_impulses, axis=1) > 0)[0]
        if fixed_mode == 'first_active' and active.size:
            fixed_impulse = cycle_impulses[int(active[0])]
        elif fixed_mode == 'first_active':
            fixed_impulse = np.zeros(xdim, dtype=float)
        else:
            raise ValueError("fixed_mode must be 'first_active'.")
        return np.repeat(fixed_impulse[None, :], n_steps, axis=0)
    raise ValueError(f'Unknown CLE mode: {mode}')


def atomic_pickle_dump(payload, output_path):
    output_path = Path(output_path)
    temporary_path = output_path.with_suffix(output_path.suffix + '.tmp')
    with temporary_path.open('wb') as file_handle:
        pickle.dump(payload, file_handle)
    temporary_path.replace(output_path)


def cache_path(kind, mode):
    return RESULTS_DIR / (
        f'cle_target_{kind}_{mode}_n{n_attractor_points_LE}'
        f'_npert{n_perturbations_LE}_nsteps{n_steps_LE}'
        f'_std{perturbation_std_LE:.0e}.pkl'
    )


def cache_metadata(kind, mode):
    return {
        'cache_version': CACHE_VERSION_LE,
        'kind': kind,
        'mode': mode,
        'sampled_cycle_ids': sampled_cycle_ids.copy(),
        'sampling_seed': sampling_seed_LE,
        'perturbation_seed': perturbation_seed_LE,
        'n_perturbations': n_perturbations_LE,
        'n_steps': n_steps_LE,
        'epsilon': perturbation_std_LE,
        'dt': dt,
        'fixed_mode': fixed_mode,
        'intervention_application': 'discrete_state_increment_after_each_free_rk4_step',
    }


def cache_is_compatible(payload, kind, mode):
    expected = cache_metadata(kind, mode)
    scalar_keys = [
        'cache_version', 'kind', 'mode', 'sampling_seed', 'perturbation_seed',
        'n_perturbations', 'n_steps', 'epsilon', 'dt', 'fixed_mode',
        'intervention_application',
    ]
    return (
        all(payload.get(key) == expected[key] for key in scalar_keys)
        and np.array_equal(
            np.asarray(payload.get('sampled_cycle_ids')),
            sampled_cycle_ids,
        )
    )


def no_renorm_distances_for_start(initial_state, perturbations, impulses):
    reference = np.asarray(initial_state, dtype=float).copy()
    states = reference[None, :] + np.asarray(perturbations, dtype=float)
    mean_distances = np.empty(len(impulses) + 1, dtype=float)
    mean_distances[0] = np.mean(np.linalg.norm(states - reference[None, :], axis=1))
    for step, impulse in enumerate(impulses):
        reference = rk4_free_step(reference) + impulse
        states = rk4_free_step(states) + impulse[None, :]
        mean_distances[step + 1] = np.mean(
            np.linalg.norm(states - reference[None, :], axis=1)
        )
    return mean_distances


def renorm_log_growth_for_start(initial_state, perturbations, impulses):
    reference = np.asarray(initial_state, dtype=float).copy()
    states = reference[None, :] + np.asarray(perturbations, dtype=float)
    mean_log_growth = np.empty(len(impulses), dtype=float)
    max_renorm_error = 0.0
    for step, impulse in enumerate(impulses):
        reference = rk4_free_step(reference) + impulse
        states_next = rk4_free_step(states) + impulse[None, :]
        deltas = states_next - reference[None, :]
        distances = np.linalg.norm(deltas, axis=1)
        safe_distances = np.clip(distances, 1e-300, None)
        mean_log_growth[step] = np.mean(
            np.log(safe_distances / perturbation_std_LE)
        )
        states = (
            reference[None, :]
            + perturbation_std_LE * deltas / safe_distances[:, None]
        )
        renorm_norms = np.linalg.norm(states - reference[None, :], axis=1)
        max_renorm_error = max(
            max_renorm_error,
            float(np.max(np.abs(renorm_norms - perturbation_std_LE))),
        )
    return mean_log_growth, max_renorm_error


def load_compatible_cache(kind, mode):
    output_path = cache_path(kind, mode)
    if not output_path.exists():
        return None, output_path
    with output_path.open('rb') as file_handle:
        payload = pickle.load(file_handle)
    if not cache_is_compatible(payload, kind, mode):
        print('Ignoring incompatible cache:', output_path)
        return None, output_path
    return payload, output_path


## Ensemble-Distance Diagnostic Without Renormalization

This diagnostic follows the initial growth of finite perturbations. It is retained for comparison with `CLE_NATURE`, but the stepwise-renormalized calculation below is the principal LLE estimate.


In [ ]:
def compute_no_renorm_mode(mode):
    kind = 'no_renorm_mean_distance'
    payload, output_path = load_compatible_cache(kind, mode)
    expected_shape = (n_attractor_points_LE, n_steps_LE + 1)
    if payload is None or np.asarray(payload.get('mean_distance_by_start')).shape != expected_shape:
        payload = cache_metadata(kind, mode)
        payload['mean_distance_by_start'] = np.full(expected_shape, np.nan)
        payload['complete'] = False

    values = np.asarray(payload['mean_distance_by_start'], dtype=float)
    completed = np.all(np.isfinite(values), axis=1)
    print(f'{mode}: resuming {int(completed.sum())}/{n_attractor_points_LE} starts')
    started = time.time()

    for point_index, cycle_id in enumerate(sampled_cycle_ids):
        if completed[point_index]:
            continue
        impulses = intervention_sequence(mode, int(cycle_id), n_steps_LE)
        values[point_index] = no_renorm_distances_for_start(
            target_array[int(cycle_id), 0],
            initial_perturbations[point_index],
            impulses,
        )
        completed[point_index] = True
        n_done = int(completed.sum())
        if n_done % checkpoint_every_LE == 0 or n_done == n_attractor_points_LE:
            payload['mean_distance_by_start'] = values
            payload['complete'] = bool(np.all(completed))
            atomic_pickle_dump(payload, output_path)
            elapsed = (time.time() - started) / 60.0
            print(f'  {mode}: {n_done}/{n_attractor_points_LE} starts ({elapsed:.2f} min)')

    payload['mean_distance_by_start'] = values
    payload['complete'] = True
    atomic_pickle_dump(payload, output_path)
    return payload


no_renorm_payloads = {}
if RUN_NO_RENORM_DIAGNOSTIC:
    for cle_mode in CLE_MODES:
        no_renorm_payloads[cle_mode] = compute_no_renorm_mode(cle_mode)


## Principal Estimate: Stepwise-Renormalized Controlled LLE

After every discrete controlled step, each separation vector is renormalized to `perturbation_std_LE`. The cache stores mean logarithmic growth per starting cycle and time step, allowing interrupted calculations to resume without storing all perturbed trajectories.


In [ ]:
def compute_renorm_mode(mode):
    kind = 'stepwise_renorm_log_growth'
    payload, output_path = load_compatible_cache(kind, mode)
    expected_shape = (n_attractor_points_LE, n_steps_LE)
    if payload is None or np.asarray(payload.get('mean_log_growth_by_start')).shape != expected_shape:
        payload = cache_metadata(kind, mode)
        payload['mean_log_growth_by_start'] = np.full(expected_shape, np.nan)
        payload['max_renorm_error_by_start'] = np.full(n_attractor_points_LE, np.nan)
        payload['complete'] = False

    growth = np.asarray(payload['mean_log_growth_by_start'], dtype=float)
    renorm_errors = np.asarray(payload['max_renorm_error_by_start'], dtype=float)
    completed = np.all(np.isfinite(growth), axis=1)
    print(f'{mode}: resuming {int(completed.sum())}/{n_attractor_points_LE} starts')
    started = time.time()

    for point_index, cycle_id in enumerate(sampled_cycle_ids):
        if completed[point_index]:
            continue
        impulses = intervention_sequence(mode, int(cycle_id), n_steps_LE)
        growth[point_index], renorm_errors[point_index] = renorm_log_growth_for_start(
            target_array[int(cycle_id), 0],
            initial_perturbations[point_index],
            impulses,
        )
        completed[point_index] = True
        n_done = int(completed.sum())
        if n_done % checkpoint_every_LE == 0 or n_done == n_attractor_points_LE:
            payload['mean_log_growth_by_start'] = growth
            payload['max_renorm_error_by_start'] = renorm_errors
            payload['complete'] = bool(np.all(completed))
            atomic_pickle_dump(payload, output_path)
            elapsed = (time.time() - started) / 60.0
            print(f'  {mode}: {n_done}/{n_attractor_points_LE} starts ({elapsed:.2f} min)')

    payload['mean_log_growth_by_start'] = growth
    payload['max_renorm_error_by_start'] = renorm_errors
    payload['complete'] = True
    atomic_pickle_dump(payload, output_path)
    return payload


renorm_payloads = {}
if RUN_STEPWISE_RENORM:
    for cle_mode in CLE_MODES:
        renorm_payloads[cle_mode] = compute_renorm_mode(cle_mode)


## LLE Summaries and Diagnostic Plots


In [ ]:
def linear_fit(x, y):
    coefficients, covariance = np.polyfit(x, y, 1, cov=True)
    fitted = coefficients[0] * x + coefficients[1]
    residual = np.sum((y - fitted) ** 2)
    total = np.sum((y - np.mean(y)) ** 2)
    r_squared = 1.0 - residual / total if total > 0 else np.nan
    return {
        'lambda': float(coefficients[0]),
        'lambda_fit_std': float(np.sqrt(covariance[0, 0])),
        'intercept': float(coefficients[1]),
        'r_squared': float(r_squared),
    }


def summarize_no_renorm(payload):
    values = np.asarray(payload['mean_distance_by_start'])
    mean_distance = np.mean(values, axis=0)
    time_axis = np.arange(values.shape[1]) * dt
    fit_slice = slice(FIT_START_LE, min(FIT_END_LE, len(time_axis)))
    summary = linear_fit(
        time_axis[fit_slice],
        np.log(np.clip(mean_distance[fit_slice], 1e-300, None)),
    )
    summary.update({
        'mode': payload['mode'],
        'method': 'ensemble_mean_distance_without_renormalization',
        'time_axis': time_axis,
        'mean_distance': mean_distance,
    })
    return summary


def summarize_renorm(payload):
    growth_by_start = np.asarray(payload['mean_log_growth_by_start'])
    mean_step_growth = np.mean(growth_by_start, axis=0)
    cumulative_growth = np.cumsum(mean_step_growth)
    time_axis = np.arange(1, len(mean_step_growth) + 1) * dt
    fit_start = max(0, FIT_START_LE)
    fit_end = min(FIT_END_LE, len(time_axis))
    summary = linear_fit(
        time_axis[fit_start:fit_end],
        cumulative_growth[fit_start:fit_end],
    )
    lambda_by_start = np.mean(
        growth_by_start[:, fit_start:fit_end],
        axis=1,
    ) / dt
    summary.update({
        'mode': payload['mode'],
        'method': 'stepwise_renormalized_discrete_controlled_map',
        'time_axis': time_axis,
        'cumulative_log_growth': cumulative_growth,
        'running_lambda': cumulative_growth / time_axis,
        'lambda_by_start': lambda_by_start,
        'lambda_sampling_std': float(np.std(lambda_by_start, ddof=1)),
        'lambda_sampling_sem': float(np.std(lambda_by_start, ddof=1) / np.sqrt(len(lambda_by_start))),
        'max_renorm_error': float(np.nanmax(payload['max_renorm_error_by_start'])),
    })
    return summary


no_renorm_summaries = {
    mode: summarize_no_renorm(payload)
    for mode, payload in no_renorm_payloads.items()
}
renorm_summaries = {
    mode: summarize_renorm(payload)
    for mode, payload in renorm_payloads.items()
}

if no_renorm_summaries:
    fig, axes = plt.subplots(1, len(no_renorm_summaries), figsize=(14, 5), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, (mode, result) in zip(axes, no_renorm_summaries.items()):
        time_axis = result['time_axis']
        mean_distance = np.clip(result['mean_distance'], 1e-300, None)
        fit_end = min(FIT_END_LE, len(time_axis))
        fitted = np.exp(result['intercept'] + result['lambda'] * time_axis[:fit_end])
        ax.semilogy(time_axis, mean_distance, color='tab:blue', label='ensemble mean distance')
        ax.semilogy(time_axis[:fit_end], fitted, color='tab:red', linestyle='--', label=f'fit λ={result["lambda"]:.4f}')
        ax.set_title(mode)
        ax.set_xlabel('time')
        ax.set_ylabel('distance')
        ax.grid(alpha=0.2)
        ax.legend()
    no_renorm_figure = FIGURES_DIR / 'cle_target_discrete_no_renorm_distances.png'
    fig.savefig(no_renorm_figure, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', no_renorm_figure)

if renorm_summaries:
    fig, axes = plt.subplots(1, len(renorm_summaries), figsize=(14, 5), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, (mode, result) in zip(axes, renorm_summaries.items()):
        ax.plot(result['time_axis'], result['running_lambda'], color='tab:purple', label='running LLE')
        ax.axhline(result['lambda'], color='tab:red', linestyle='--', label=f'fit λ={result["lambda"]:.4f}')
        ax.set_title(mode)
        ax.set_xlabel('time')
        ax.set_ylabel('LLE')
        ax.grid(alpha=0.2)
        ax.legend()
    renorm_figure = FIGURES_DIR / 'cle_target_discrete_stepwise_renorm.png'
    fig.savefig(renorm_figure, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', renorm_figure)

print('Stepwise-renormalized results:')
for mode, result in renorm_summaries.items():
    print(
        f"  {mode}: λ={result['lambda']:.6f}, "
        f"fit_std={result['lambda_fit_std']:.6f}, "
        f"sampling_SEM={result['lambda_sampling_sem']:.6f}, "
        f"R²={result['r_squared']:.6f}"
    )


## Save Compact Controlled-LLE Results


In [ ]:
results_payload = {
    'experiment_name': EXPERIMENT_NAME,
    'cse_summary': cse_summary,
    'methodology': (
        'prescribed_discrete_state_increments_applied_after_free_rk4_step_'
        'to_reference_and_perturbed_states'
    ),
    'closed_loop_linearization': False,
    'cache_version': CACHE_VERSION_LE,
    'configuration': {
        'n_attractor_points': n_attractor_points_LE,
        'n_perturbations': n_perturbations_LE,
        'n_steps': n_steps_LE,
        'perturbation_std': perturbation_std_LE,
        'fit_start': FIT_START_LE,
        'fit_end': FIT_END_LE,
        'sampling_seed': sampling_seed_LE,
        'perturbation_seed': perturbation_seed_LE,
        'sampled_cycle_ids': sampled_cycle_ids,
    },
    'max_target_reconstruction_error': max_reconstruction_error,
    'no_renorm_summaries': no_renorm_summaries,
    'renorm_summaries': renorm_summaries,
    'cache_files': {
        kind: {
            mode: str(cache_path(kind, mode))
            for mode in CLE_MODES
        }
        for kind in ('no_renorm_mean_distance', 'stepwise_renorm_log_growth')
    },
}

results_file = RESULTS_DIR / 'cle_target_xyz_control_discrete_results.pkl'
atomic_pickle_dump(results_payload, results_file)
print('Saved compact results:', results_file)
print('Maximum TARGET reconstruction error:', max_reconstruction_error)
